# HFT高频策略工厂类 IHftStrategyFact
```cpp
class WtHftStraFact : public IHftStrategyFact
```
该类实现了 IHftStrategyFact 接口，是HFT策略的工厂类。负责创建、管理和删除HFT策略实例，支持策略的动态加载和插件化开发。

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称的实现
 * @return const char* 返回策略工厂的名称字符串
 * 
 * 该函数返回策略工厂的名称，用于标识和管理不同的策略工厂。
 * 返回的名称是常量字符串"WtHftStraFact"，在系统中应该是唯一的。
 * 
 * @note 返回的是常量字符串指针，不需要调用者释放内存
 */
const char* WtHftStraFact::getName()
{
	return FACT_NAME;
}
```

## 枚举策略名称 enumStrategy
```cpp
/**
 * @brief 枚举策略名称的实现
 * @param cb 枚举策略名称的回调函数，每枚举到一个策略都会调用此回调
 * 
 * 该函数枚举工厂中所有可用的策略类型，通过回调函数通知调用者。
 * 当前工厂支持的策略：
 * - "SimpleHft": SimpleHft简单高频交易策略示例
 * 
 * 回调函数会被调用一次，传入以下参数：
 * - factName: 工厂名称（"WtHftStraFact"）
 * - straName: 策略名称（"SimpleHft"）
 * - isLast: 是否为最后一个策略（true，因为只有一个策略）
 * 
 * @note 如果将来添加更多策略，需要在此函数中添加更多的回调调用
 */
void WtHftStraFact::enumStrategy(FuncEnumHftStrategyCallback cb)
{
	cb(FACT_NAME, "SimpleHft", true); // 调用回调函数，传入工厂名称、策略名称和是否为最后一个策略
}
```

## 创建策略实例 createStrategy
```cpp
/**
 * @brief 创建策略实例的实现
 * @param name 策略名称，用于指定要创建的策略类型
 * @param id 策略唯一标识符，用于在系统中唯一标识该策略实例
 * @return HftStrategy* 返回创建的策略对象指针，如果策略名称不存在则返回NULL
 * 
 * 该函数根据策略名称创建对应的策略对象实例。
 * 当前支持的策略类型：
 * - "SimpleHft": 创建SimpleHft简单高频交易策略实例
 * 
 * 如果传入的策略名称不匹配任何已知策略，则返回NULL。
 * 
 * @note 调用者负责管理返回的指针，使用完毕后应通过deleteStrategy删除
 */
HftStrategy* WtHftStraFact::createStrategy(const char* name, const char* id)  // 创建策略函数实现
{
	if(strcmp(name, "SimpleHft") == 0) // 比较策略名称是否为"SimpleHft"
	{
		return new WtHftStraDemo(id); // 创建SimpleHft策略实例并返回
	}
	return NULL;
}
```

## 删除策略实例 deleteStrategy
```cpp
/**
 * @brief 删除策略实例的实现
 * @param stra 要删除的策略对象指针
 * @return bool 删除成功返回true，失败返回false
 * 
 * 该函数删除指定的策略对象，释放相关资源。
 * 删除前会进行以下检查：
 * 1. 检查策略指针是否为空，如果为空则直接返回true（视为成功）
 * 2. 检查策略是否属于本工厂创建，通过比较策略的工厂名称
 * 3. 只有属于本工厂的策略才会被删除，其他策略返回false
 * 
 * @note 删除后策略指针将失效，调用者不应再使用该指针
 */
bool WtHftStraFact::deleteStrategy(HftStrategy* stra)
{
	if (stra == NULL)
		return true;

	if (strcmp(stra->getFactName(), FACT_NAME) != 0) // 检查策略是否属于本工厂创建
		return false; // 如果不属于本工厂，返回false（拒绝删除）

	delete stra; // 删除策略对象，调用析构函数释放资源
	return true;
}
```

# 简单高频交易策略示例类 WtHftStraDemo
```cpp
class WtHftStraDemo : public HftStrategy
```
实现了一个基于Tick数据的简单高频交易策略，展示了HFT策略的基本开发模式。

## 成员
* **核心上下文与数据**
  * `IHftStraCtx* _ctx`：HFT策略上下文对象指针
    * 用于访问数据接口（如获取Tick）和交易接口（如下单、撤单）
  * `WTSTickData* _last_tick`：最后接收到的Tick数据指针
    * 用于保存最新的行情快照（注：代码注释提到当前未使用，保留以备将来扩展）
* **策略配置参数**
  * `std::string _code`：合约代码。策略交易的目标合约
  * `uint32_t _secs`：订单超时时间（秒）。超过此时间未成交的订单会被自动撤销
  * `uint32_t _freq`：交易频率限制（毫秒）。两次交易操作之间的最小时间间隔
  * `int32_t _offset`：价格偏移跳数。下单价格相对于最新价的偏移量（正数向上，负数向下）
  * `uint32_t _unit`：交易单位。交易的数量单位（如股票为100，期货为1）
  * `double _reserved`：基础持仓量。
    * 用于持仓修正，计算逻辑为：实际持仓 = 查询持仓 - 基础持仓
  * `bool _stock`：品种模式标记
  * `true`表示股票模式（不支持做空），`false`表示期货模式
* **订单管理与并发控制**
  * `IDSet _orders`：订单ID集合
    * typedef std::unordered_set<uint32_t> IDSet;
    * 存储当前策略管理的所有未完成订单的本地ID
  * `std::mutex _mtx_ords`：订单集合互斥量
    * 用于多线程环境下保护 `_orders` 集合的并发访问安全
  * `uint32_t _cancel_cnt`：撤销订单计数器
    * 记录撤销的订单数量（主要用于统计和调试）
* **运行时状态与流控**
  * `uint64_t _last_entry_time`：最后交易时间戳
    * 单位：微秒，用于结合 `_freq` 进行交易频率控制
  * `bool _channel_ready`：交易通道状态标记
    * `true`表示通道就绪可交易，`false`表示通道丢失或未就绪
  * `uint32_t _last_calc_time`：最后计算时间
    * 单位：分钟，用于控制计算频率（在 `do_calc` 逻辑中使用）

## 基本属性

### 获取策略名称 getName
```cpp
/**
 * @brief 获取策略名称的实现
 * @return const char* 返回策略的名称字符串
 * 
 * 该函数返回策略的名称，用于标识策略类型。
 * 返回值为"HftDemoStrategy"，表示这是HFT策略示例。
 */
const char* WtHftStraDemo::getName()
{
	return "HftDemoStrategy"; // 返回策略名称字符串
}
```

### 获取工厂名称 getFactName
```cpp
/**
 * @brief 获取所属策略工厂名称的实现
 * @return const char* 返回策略所属的工厂名称字符串
 * 
 * 该函数返回策略所属的策略工厂名称，用于标识策略的来源工厂。
 * 返回值为"WtHftStraFact"，表示该策略由WtHftStraFact工厂创建。
 * 
 * @note FACT_NAME常量定义在WtHftStraFact.cpp中，值为"WtHftStraFact"
 */
const char* WtHftStraDemo::getFactName()
{
	return FACT_NAME;
}
```

## 生命周期与初始化

### 策略参数初始化 init
```cpp
/**
 * @brief 策略初始化实现
 * @param cfg 策略配置参数，包含策略运行所需的所有参数
 * @return bool 初始化成功返回true，失败返回false
 * 
 * 该函数从配置参数中加载策略运行所需的参数，包括：
 * - code: 合约代码，策略交易的合约（必需参数）
 * - second: 订单超时时间（秒），超过此时间未成交的订单会被撤销（必需参数）
 * - freq: 交易频率限制（毫秒），两次交易之间的最小时间间隔（必需参数）
 * - offset: 价格偏移跳数，下单价格相对于最新价的偏移（必需参数）
 * - reserve: 基础持仓量，用于持仓修正（可选参数，默认为0）
 * - stock: 是否为股票，true表示股票模式，false表示期货模式（可选参数，默认为false）
 * 
 * @note 如果配置参数为空或缺少必要参数，可能导致运行时错误
 */
bool WtHftStraDemo::init(WTSVariant* cfg)  // 策略初始化函数实现
{
	_code = cfg->getCString("code");
	_secs = cfg->getUInt32("second");
	_freq = cfg->getUInt32("freq");
	_offset = cfg->getUInt32("offset");
	_reserved = cfg->getDouble("reserve");
	_stock = cfg->getBoolean("stock");
	_unit = _stock ? 100 : 1;

	return true;
}
```

### 策略初始化完成回调 on_init
策略生命周期的**初始化阶段**。在策略实例创建并加载参数后调用，用于准备运行环境。
* **数据可用性验证**
  * 调用 `ctx->stra_get_bars` 预读取指定代码的 1 分钟 K 线数据。
  * **目的**：验证底层数据接口是否通畅，以及历史数据是否存在。
  * **释放**：读取后立即 `release()`，因为此处仅作检查，不持有数据。
* **订阅实时行情**
  * 调用 `ctx->stra_sub_ticks` 订阅目标合约的 Tick 数据流。
  * **关键点**：只有订阅了 Tick，后续的 `on_tick` 才会被引擎触发。
* **上下文保存**
  * 将传入的 `IHftStraCtx* ctx` 保存到成员变量 `_ctx` 中，供策略其他非回调函数（如 `check_orders`）使用。
```cpp
/**
 * @brief 策略初始化完成回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @note 该函数重写了HftStrategy基类的纯虚函数
 */
void WtHftStraDemo::on_init(IHftStraCtx* ctx)
```

## 行情事件回调

### Tick数据回调 on_tick
高频数据驱动入口。这是 HFT 策略的主脉搏，每当有新的 Tick 到达时触发，负责调度风控、订单管理和核心计算。
* **合约过滤**
  * 检查 `newTick` 的代码是否与策略配置的 `_code` 一致，若不一致直接忽略。
* **优先处理未完成订单**
  * **判断**：如果本地订单集合 `_orders` 不为空（说明有未完结的挂单）。
  * **动作**：调用 `check_orders()` 检查订单是否超时。
  * **阻断**：直接 `return`。
  * **设计意图**：**状态机保护**。在高频交易中，如果手头有未完成的挂单，通常不进行新的计算和下单，防止逻辑混乱或资金占用超限。
* **通道状态检查**
  * 如果交易通道未就绪 `!_channel_ready`，直接返回，确保不向断开的通道发送指令。
* **核心逻辑执行**
  * 如果以上检查通过，调用 `do_calc(ctx)` 执行具体的信号计算和下单逻辑。
```cpp
/**
 * @brief Tick数据处理回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @param code 标准合约代码，触发Tick数据的合约
 * @param newTick 新的Tick数据，包含最新的价格和成交量信息
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_tick(IHftStraCtx* ctx, const char* code, WTSTickData* newTick)
``

### K线闭合回调 on_bar
```cpp
/**
 * @brief K线闭合回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @param code 标准合约代码，触发K线数据的合约
 * @param period K线周期，如"m1"、"m5"等
 * @param times K线倍数
 * @param newBar 新的K线数据
 * 
 * 该函数在K线闭合时被调用，用于处理K线数据。
 * SimpleHft策略主要基于Tick数据，因此K线数据处理为空实现。
 * 
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_bar(IHftStraCtx* ctx, const char* code, const char* period, uint32_t times, WTSBarStruct* newBar) {}
```

## 交易回报与状态

### 订单状态回报 on_order
订单状态更新回调。当发出的订单有状态变化（成交、撤单、部分成交）时触发，用于维护策略内部的订单状态机。
* **身份验证**
  * 在 `_orders` 集合中查找回调的 `localid`。
  * 如果找不到，说明该订单不是本策略发出的（或者是重启前遗留的已清理订单），直接忽略。
* **终态处理**
  * **条件**：如果订单已撤销 (`isCanceled`) 或 剩余数量为0 (`leftQty == 0`，即完全成交)。
  * **并发安全操作**：
    * 获取锁 `_mtx_ords`。
    * 从 `_orders` 集合中移除该订单 ID。
    * 如果是撤单，更新统计计数器 `_cancel_cnt`。
    * 释放锁。
* **事件驱动重算**
  * **动作**：调用 `do_calc(ctx)`。
  * **设计意图**：订单结束意味着持仓发生了变化（成交）或资金/挂单额度被释放（撤单）。这是一个**状态变更事件**，策略应立即根据最新的仓位和行情重新评估是否需要补单或反手，而不是被动等待下一个 Tick。
```cpp
/**
 * @brief 订单回报回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码
 * @param isBuy 是否为买入，true表示买入，false表示卖出
 * @param totalQty 订单总数量
 * @param leftQty 订单剩余数量
 * @param price 订单价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 * @param userTag 用户标签，用于标识订单来源
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_order(IHftStraCtx* ctx, uint32_t localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled, const char* userTag)
```

### 成交回报 on_trade
```cpp
/**
 * @brief 成交回报回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码
 * @param isBuy 是否为买入，true表示买入，false表示卖出
 * @param qty 成交数量
 * @param price 成交价格
 * @param userTag 用户标签，用于标识订单来源
 * 
 * 该函数在订单成交时被调用，用于处理成交回报。
 * 主要功能：
 * 1. 记录成交信息
 * 2. 触发策略重新计算，根据新的持仓情况生成交易信号
 * 
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_trade(IHftStraCtx* ctx, uint32_t localid, const char* stdCode, bool isBuy, double qty, double price, const char* userTag)
{
	do_calc(ctx); // 执行策略计算逻辑，根据新的持仓情况生成交易信号
}
```

### 持仓变化回报 on_position
```cpp
/**
 * @brief 持仓回报回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @param stdCode 标准合约代码
 * @param isLong 是否为多头，true表示多头，false表示空头
 * @param prevol 变化前的持仓量
 * @param preavail 变化前的可用持仓量
 * @param newvol 变化后的持仓量
 * @param newavail 变化后的可用持仓量
 * 
 * 该函数在持仓发生变化时被调用，用于处理持仓回报。
 * SimpleHft策略主要关注持仓变化，但当前实现为空，可以根据需要扩展。
 * 
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_position(IHftStraCtx* ctx, const char* stdCode, bool isLong, double prevol, double preavail, double newvol, double newavail) {}
```

### 委托结果回报 on_entrust
```cpp
/**
 * @brief 委托回报回调实现
 * @param localid 本地订单ID，用于标识订单
 * @param bSuccess 委托是否成功，true表示成功，false表示失败
 * @param message 委托结果消息
 * @param userTag 用户标签，用于标识订单来源
 * 
 * 该函数在委托回报时被调用，用于处理委托结果。
 * SimpleHft策略当前实现为空，可以根据需要扩展处理逻辑。
 * 
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_entrust(uint32_t localid, bool bSuccess, const char* message, const char* userTag) {}
```

## 交易通道事件

### 通道就绪回调 on_channel_ready
**交易通道就绪/连接恢复的处理钩子**。它不仅是简单的连接确认，更承担着**状态一致性检查**的关键职责。

* **场景**：策略启动时，或者盘中网络断开后重新连接成功时。
* **核心任务**：确保策略内部记录的订单状态与交易所/柜台的实际挂单状态一致，防止“幽灵订单”导致仓位失控。

**具体过程**：

* **获取柜台状态**
  * 调用 `_ctx->stra_get_undone` 查询该合约在柜台/交易网关当前的**未完成挂单总量** `undone`。
* **一致性校验**
  * **判断条件**：如果 `undone != 0`（柜台有挂单）**并且** `_orders` 为空（策略本地内存认为没挂单）。
  * **含义**：这通常意味着策略发生了重启（丢失了内存状态），或者有外部程序干预了挂单。此时这些挂单处于“失管”状态。
* **接管与清理**
  * **全部撤单**：策略发出指令 `_ctx->stra_cancel` 撤销这些不在管理范围内的订单。
  * **接管 ID**：虽然发出了撤单，但策略仍将返回的订单 ID 插入本地 `_orders` 集合。
  * **目的**：为了能够接收后续的“撤单成功”回报，从而正确地维护生命周期。如果不接管，后续的 `on_order` 回报会被丢弃。
* **开启交易权限**
  * 设置 `_channel_ready = true`，允许 `on_tick` 开始驱动核心逻辑。
```cpp
/**
 * @brief 交易通道就绪回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_channel_ready(IHftStraCtx* ctx)
```

### 通道断开回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 * 该函数在交易通道丢失时被调用，表示无法进行交易。
 * @note 该函数重写了HftStrategy基类的虚函数
 */
void WtHftStraDemo::on_channel_lost(IHftStraCtx* ctx)
{
	_channel_ready = false;
}
```

## 内部核心逻辑

### 订单检查与超时处理 check_orders
**订单生命周期管理（超时风控）**。在高频交易中，挂单如果不能在极短时间内成交，通常意味着市场微观结构已经改变，原本的价格优势不再，因此必须撤单。

* **机制**：**时间加权清理**。
* **调用时机**：通常在 `on_tick` 的最开始调用，即在产生新信号之前，先清理旧战场的遗留物。

**具体过程**：
* **前置检查**
  * 如果本地没有挂单 `_orders.empty()`，或者从未交易过，直接返回。
* **计算时间差**
  * 获取当前时间戳 `now`（微秒级）。
  * 计算滞留时间：`elapsed = now - _last_entry_time`。
* **超时判定**
  * 如果 `elapsed >= _secs * 1000`（超过了配置的 N 秒）：
    * **批量撤单**：加锁 `_mtx_ords`，遍历 `_orders` 集合中的所有 ID，逐一调用 `_ctx->stra_cancel`。
    * **统计更新**：增加 `_cancel_cnt` 计数（用于监控策略撤单率，撤单率过高可能在某些交易所被封禁）。
    * **日志记录**：记录超时撤单事件。
```cpp
/**
 * @brief 检查订单状态的实现
 */
void WtHftStraDemo::check_orders()
```

### 核心计算与信号生成 do_calc
**核心量化逻辑引擎**。它负责信号计算、仓位控制和具体的下单执行。

* **逻辑流**：**流控 -> 信号计算 -> 状态机判定 -> 执行**。

**具体过程**：

* **交易频率流控**
  * 检查距离上次开仓时间 `_last_entry_time` 是否小于 `_freq`（毫秒）。如果是，直接返回，防止高频开仓打爆账户或超过柜台限制。
* **计算微观理论价**
  * 公式：`pxInThry = (BidP * AskQ + AskP * BidQ) / (BidQ + AskQ)`
  * **原理**：利用买一/卖一量的加权不平衡（Imbalance）来预测中间价的偏移。
    * 如果买量巨大（BidQ 高），理论价向卖一价（AskP）靠近 -> 倾向做多。
    * 如果卖量巨大（AskQ 高），理论价向买一价（BidP）靠近 -> 倾向做空。
* **信号生成**
  * **正向信号 (Signal = 1)**：理论价 `pxInThry` > 当前最新价 `price`。
  * **反向信号 (Signal = -1)**：理论价 `pxInThry` < 当前最新价 `price`。
* **执行状态机**
  * 获取当前净持仓 `curPos` 并减去底仓 `_reserved`。
  * **做多分支**：
    * 条件：信号为正 **且** 当前为空仓或平仓状态 (`curPos <= 0`)。
    * 动作：以 `最新价 + _offset` 的价格发出买单。
  * **做空分支**：
    * 条件：信号为负 **且** (当前持多仓 **或** (允许做空 **且** 持仓为0))。
    * 动作：以 `最新价 - _offset` 的价格发出卖单。
* **状态更新**
  * 发出指令后，立即将返回的 `ids` 加入 `_orders` 集合进行监控。
  * 更新 `_last_entry_time` 以重置流控计时器。
```cpp
/**
 * @brief 执行策略计算逻辑的实现
 * @param ctx HFT策略上下文对象，提供数据访问和交易执行接口
 */
void WtHftStraDemo::do_calc(IHftStraCtx* ctx)
```